# Midiendo Asimetrías Materia-Antimateria en el Gran Colisionador de Hadrones

Esta versión del cuaderno se ejecuta en Google Colab, pero primero necesitas instalar algunos paquetes

In [ ]:
# pip install uproot awkward awkward-pandas

Descarga el archivo [open-data-project-1.1.tar.gz](https://github.com/matchuf/PWF_Gt-Hn-CR_LHCb_open-data), descomprímelo y guárdalo en tu Google Drive.

Además, tienes que descargar los archivos de datos reales: [B2HHH_MagnetUp.root](https://drive.google.com/file/d/10FGQvnZBjzQ4F0d_ZfLzTeoZjKD1IGl0/view?usp=sharing) y [B2HHH_MagnetDown.root](https://drive.google.com/file/d/1meKf8X6jOc176WJOasT21tcITngCRjXh/view?usp=sharing) y guárdalos en la carpeta `YOUR_DRIVE_PATH/opendata-project-1.1/Data/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/<your-folder>/opendata-project-1.1")

# Check that you are in the correct path
!ls

# Introducción
____

<b> ¡Bienvenidos al primer proyecto guiado del Portal de Datos Abiertos de LHCb - Hackathon Centroamérica! </b>

<div align="justify">Aquí podrás analizar datos tomados por el Gran Colisionador de Hadrones (LHC) en el CERN. El objetivo de este estudio es que puedas buscar diferencias en el comportamiento de la materia y la  <a href="https://en.wikipedia.org/wiki/Antimatter">antimateria</a>. Este proyecto te permitirá realizar tu propio análisis de datos a un nivel similar al de la investigación del CERN. Este proyecto no requiere un conocimiento detallado de la física de partículas. Es más adecuado para personas con una formación científica y matemática equivalente a la requerida para solicitar ingreso a la universidad en una disciplina de ciencias, tecnología, ingeniería o matemáticas. También sería ventajoso tener cierta familiaridad previa con la programación informática. Se proporciona información teórica adicional o conocimientos de programación que puedas necesitar a medida que avanza el proyecto.</div>

Antes de empezar, puede ser útil que aprendas más sobre las asimetrías materia-antimateria, qué esperamos aprender al estudiarlas y cómo podemos detectarlas con experimentos como el experimento LHCb en el CERN.

Aquí hay algunos detalles que se relacionan directamente con este proyecto:

* ¿Cuál es el enfoque de la [física de partículas](Background-Information-Notebooks/ProjectIntro.ipynb) de este experimento? ¿y qué estudiaré en este proyecto?

* ¿Cómo registra los datos el [detector](Background-Information-Notebooks/DetectorSoftwareDataSample.ipynb) LHCb?

# Empezando

## Objetivos

* Familiarizar con la ayuda disponible para la programación.
* Introducción de los datos de la simulación en el programa.
________________________

<div align="justify">Al igual que los investigadores del CERN, programarás tu propio análisis. Esto utilizará el lenguaje de programación Python. No es necesario tener experiencia previa en programación en Python para seguir este proyecto. Habrá pistas disponibles para ayudarte en el camino. Puedes encontrar útiles estos tutoriales sobre Python:</div>

[Tutoriales de Python](http://www.tutorialspoint.com/python/)

La guía de programación más importante que te proporcionamos está en forma de un análisis no relacionado. Hemos realizado un análisis de los ganadores del Premio Nobel [Example-Analysis.ipynb](Example-Analysis.ipynb). Ese enlace te proporciona el código completo para ello. Las habilidades de programación requeridas para este análisis de los Nobel son muy similares a las necesarias para el análisis de física de partículas. Por lo tanto, al leer y comprender ese análisis, puedes copiar y adaptar las líneas de código para realizar tu análisis de física de partículas.

## Reading simulation data

Para empezar y comprobar que el primer código que escribirás funciona correctamente, es mejor empezar analizando datos simulados en lugar de datos reales del LHC. Los datos reales contienen no solo el tipo de eventos que deseas analizar, conocidos como 'señal', sino también eventos que pueden falsificarlos, conocidos como 'fondo'. Las mediciones de datos reales también están limitadas por la resolución del detector. Los datos de simulación simplificados proporcionados aquí contienen solo los eventos de señal y proporcionan los resultados que se obtendrían para un detector perfecto.

IMPORTANTE: Para cada recuadro de código que ya tenga código, como el siguiente, debes hacer clic dentro y presionar shift+enter para ejecutar el código.

Si el `In [x]`: a la izquierda de un recuadro de código cambia a `In [*]`:, significa que el código en ese recuadro se está ejecutando actualmente.

In [ ]:
# Importing  necessary libraries
import numpy
import os
import uproot
import pandas
from matplotlib import pyplot as plt

Si necesitas ayuda con la programación, además del [código de ejemplo](Example-Analysis.ipynb), hay algunas pistas dentro de cada sección y una [lista de referencia de funciones](Background-Information-Notebooks/FunctionReferences.pdf).

In [ ]:
# Let us now load the simulated data as data frame
with uproot.open("Data/PhaseSpaceSimulation.root:PhaseSpaceTree") as data:
    sim_data = data.arrays(data.keys(),library="pd")

<div align="justify">Ahora que puedes acceder a los datos, puedes usar varias funciones que te ayudarán a analizarlos. Puedes encontrar estas funciones en las bibliotecas al principio de la página. Intenta hacer una tabla de parte de la información dentro de tu archivo de datos para que puedas hacerte una idea de los valores típicos de los datos en el conjunto. Comprender el rango de valores para diferentes variables te ayudará a trazar gráficos.</div>

Los datos contienen información sobre 'eventos' que se observaron en el detector. Un evento se refiere a las partículas producidas cuando tuvo lugar una interacción al chocar dos protones en el LHC. Los datos que tienes incluyen información sobre partículas observadas en el detector después de cada colisión. Si piensas en los datos como una tabla, cada fila de la tabla son los resultados de una colisión diferente. Las columnas de la tabla son diferentes cantidades medidas sobre las partículas producidas en la colisión.

Nos interesa analizar las desintegraciones de partículas llamadas mesones B<sup>+</sup> o B<sup>-</sup> que se desintegran en otros tres mesones llamados kaones (K<sup>+</sup> o K<sup>-</sup>). Los eventos que se te han dado son aquellos en los que este proceso pudo haber ocurrido. El detector se ha utilizado para reconstruir trayectorias que podrían provenir de los kaones. Se te dan los momentos medidos, la carga y la probabilidad de que las trayectorias sean kaones. Se te da información para tres trayectorias en cada evento, aquellas que podrían ser los tres kaones en los que se desintegró un mesón B<sup>+</sup> o B<sup>-</sup>. La siguiente información está disponible sobre cada evento: [lista de información](Background-Information-Notebooks/EventData.ipynb)

In [ ]:
# make a table of the data variables here
sim_data.head()

# Reconstrucción de Masa Invariante

## Objetivos
* Trazar un histograma del momento de uno de los candidatos a kaón.
* Calcular la energía de cada uno de los candidatos a kaón.
* Trazar las masas invarinates de los mesones B<sup>+</sup> o B<sup>-</sup>.
____

### Trazando una característica:

Puedes trazar cualquier característica de los datos en un histograma. Elige cualquier intervalo (binning) adecuado que te permita observar claramente la distribución de la variable. Intenta hacer un histograma para la componente x del momento del primer candidato a kaón (H1_PX):

In [ ]:
# make a histogram of the H1_PX variable here

El momento es una cantidad **vectorial**, tiene componentes x, y, z. Intenta calcular la **magnitud** del momento del primer candidato a kaón y trazar un histograma de esto; necesitarás las variables `H1_PX`, `H1_PY` y `H1_PZ`.

In [ ]:
# calculate a variable for the magnitude of the momentum of the first kaon
# plot a histogram of this variable

### Hints

**Magnitud de un vector**: La magnitud al cuadrado de un vector está dada por la suma de los cuadrados de sus componentes en las direcciones x, y y z: $p^2 = p_x^2+p_y^2+p_z^2$, donde $p$ es la magnitud del momento, y $p_x,p_y,p_z$ son las componentes del momento en las direcciones X, Y y Z.

## Energía y Masa

La teoría de la relatividad especial de Einstein relaciona la energía, la masa y el momento. Hemos medido el momento de los candidatos a kaón en el detector, y acabamos de trazar una de las componentes del momento del kaón, así como la magnitud del momento. La masa invariante del kaón es bien conocida y puedes consultarla. Deseamos determinar la energía de los kaones.

Aquí hay una breve guía sobre la relación energía-momento de la [relatividad especial](Background-Information-Notebooks/SpecialRelativity.ipynb). Puedes encontrar más información en las páginas de Wikipedia sobre [Masa invariante](https://en.wikipedia.org/wiki/Invariant_mass) y la Relación [energía-momento](https://en.wikipedia.org/wiki/Energy%E2%80%93momentum_relation).

Ahora, calcula la energía del primer candidato a kaón usando:

<center> $E^2 = p^2 + m^2$ </center>

In [ ]:
# calculate the energy of the first kaon

In [ ]:
# plot a histogram of this variable

### Hints

**Cálculo de la energía** - Usa la variable de magnitud del momento que calculaste arriba y la masa invariante conocida del kaón para calcular la energía del primer hadrón. Calcula la energía al cuadrado, y luego la energía y traza esto.

**Masa del kaón** - Puedes encontrar la masa del kaón en Wikipedia o en libros de texto de física. También hay una referencia utilizada por los físicos de partículas: todo nuestro conocimiento de las propiedades de las partículas está recopilado por el Particle Data Group [aquí](http://pdg.lbl.gov/2014/reviews/rpp2014-rev-charged-kaon-mass.pdf).

Calculate the momenta and energies of the second and third kaon candidates also.


In [ ]:
# calculate variables for the energy of the other two kaons

## Añadiendo características del mesón $B$
En este análisis, buscamos mesones B<sup>+</sup> o B<sup>-</sup> (ver [mesón B](https://en.wikipedia.org/wiki/B_meson)) que se hayan desintegrado en tres [kaones](https://en.wikipedia.org/wiki/Kaon) cargados.

La energía es una cantidad que se conserva. Esto significa que puedes usar la energía de los tres kaones "hijos", que calculaste anteriormente, para calcular la energía que debió tener el mesón B que se desintegró en ellos.

El momento también es una cantidad que se conserva. Por lo tanto, también puedes usar los momentos de los kaones "hijos" para calcular el momento del mesón B. Pero ten cuidado: el momento es una cantidad *vectorial*.

Usando la energía del mesón B y la magnitud del momento del mesón B, puedes usar nuevamente la relación energía-momento. Esta vez la estás aplicando al mesón B. Esto te permitirá calcular la masa invariante del mesón B.

In [ ]:
# calculate the energy of the B meson

In [ ]:
# calculate the momentum components of the B meson
# and the magnitude of the momentum of the B meson

In [ ]:
# calculate the B meson invariant mass
# plot the B meson invariant mass in a histogram

Deberías obtener un gráfico que tenga un pico pronunciado en la masa del mesón B<sup>+</sup>. La masa del mesón B<sup>+</sup> y del B<sup>-</sup> son iguales. Verifica que el pico de tu gráfico esté en la [masa conocida](http://pdg.lbl.gov/2014/listings/rpp2014-list-B-plus-minus.pdf) del mesón B. **¡Felicitaciones!**

Recuerda que has hecho este gráfico para datos simulados. ¿Cómo crees que se verían diferentes los gráficos para datos reales? En la siguiente sección comenzarás a trabajar con los datos reales del LHC.

### Hint

**Energía del mesón B** - Por conservación de la energía, la energía del mesón B será la suma de las energías de los tres kaones: $E_B=E_{K1}+E_{K2}+E_{K3}$, donde $E_B$ es la energía del mesón B, $E_{K1}, E_{K2}, E_{K3}$ son las energías de cada uno de los kaones.

**Momento del mesón B** - Por conservación del momento, la componente X del momento del mesón B será la suma de las componentes X del momento de los tres kaones: $px_B=px_{K1}+px_{K2}+px_{K3}$, donde $px$ es la componente en la dirección X del momento del mesón B, $px_{K1},px_{K2},px_{K3}$ son las componentes en la dirección X de los momentos de los tres kaones. Luego puedes hacer lo mismo con las componentes Y y Z. Una vez obtenidas las componentes X, Y y Z del momento del B, puedes encontrar la magnitud del momento del mesón B.

**Masa invariante del mesón B** - Reorganiza la ecuación $E^2=p^2+m^2$ para encontrar $m^2$. Usando los valores de la magnitud del momento del mesón B y la energía del mesón B, encuentra la masa del mesón B.

**Trazado del histograma** - Asegúrate de que el rango de tu gráfico de masa esté configurado adecuadamente para que puedas ver el pico de masa. Una vez que hayas encontrado el pico, puedes ajustar el rango apropiadamente. No es necesario que comiences tu gráfico en una masa de 0.

**Unidades** - Los datos que se te proporcionan tienen energías en 'MeV' (10<sup>6</sup> electronvoltios). La masa del mesón B suele expresarse en 'GeV/c<sup>2</sup>' (10<sup>9</sup> electronvoltios).

# Trabajando con datos reales y aplicando cortes (cuts)

## Objetivos - Desafíos 1 y 2
* Filtrar los datos que no provienen del canal B<sup>+</sup> → K<sup>+</sup>K<sup>+</sup>K<sup>−</sup>, o su equivalente de antipartícula B<sup>-</sup> → K<sup>+</sup>K<sup>-</sup>K<sup>−</sup>

* Trazar un histograma de la masa del mesón B para los datos reales y observar cómo los diferentes cortes afectan los datos

En la sección anterior has analizado los datos de simulación para determinar la masa invariante del mesón B. Ahora puedes empezar a aplicar los métodos que has utilizado a los datos reales de LHCb. Estos datos fueron recolectados por el detector LHCb en el CERN durante 2011, el primer año importante de operaciones del LHC.

Los datos que se te han dado han sido filtrados para seleccionar solo eventos que probablemente provengan de mesones B<sup>+</sup> o B<sup>-</sup> desintegrándose en tres partículas finales cargadas. Te interesa el caso en el que estas tres partículas finales son kaones cargados K<sup>+</sup> o K<sup>-</sup>.

Se ha proporcionado una introducción sobre el [detector y la muestra de datos](Background-Information-Notebooks/DetectorSoftwareDataSample.ipynb). Como información de contexto, también proporcionamos más información sobre la [selección](Background-Information-Notebooks/DataSelection.ipynb) que se ha aplicado para seleccionar esta muestra de datos.

## Preselección
Quieres aplicar una preselección a las tres trayectorias del estado final que:
* Asegure que no son muones [ej. `(H1_isMuon==0)`, y similarmente para `H2` y `H3`]
* Requiera que cada una tenga una baja probabilidad de ser piones [ej. `(H1_ProbPi) es una variable de identificación de partículas - probabilidad de H1 de ser un pion`]
* Requiera que cada una tenga una alta probabilidad de ser un kaón [ej. `(H1_ProbK) es una variable de identificación de partículas - probabilidad de H1 de ser un kaón`]

Necesitas encontrar un equilibrio entre hacer cortes demasiado laxos que incluyan demasiados eventos de fondo y cortes demasiado estrictos que rechacen muchos de tus eventos de señal.

Para encontrar los cortes de selección adicionales más adecuados, familiarízate con [cómo los cortes pueden afectar la significancia del resultado final](Background-Information-Notebooks/CutsInformation.ipynb). No dudes en volver a esta etapa más tarde y ajustar tus cortes para ver el impacto.

La preselección que crees se aplicará si le das el nombre 'preselection'.

Hemos proporcionado un ejemplo de preselección en las pistas, así que siéntete libre de usarlo para empezar si lo deseas. Comienza con una preselección laxa y luego refínala después de haber estudiado los gráficos.

In [ ]:
# Make your preselection here, this line applies no preselection

La siguiente línea de código simplemente carga los datos reales en un nuevo DataFrame; esto puede tardar unos minutos.
También aplica la preselección que has creado si la llamaste preselection.

In [ ]:
with uproot.open("Data/B2HHH_MagnetUp.root:DecayTree") as data:
    real_data = data.arrays(data.keys(),preselection,library="pd")

with uproot.open("Data/B2HHH_MagnetDown.root:DecayTree") as data:
    real_data._append(data.arrays(data.keys(),preselection,library="pd"), ignore_index=True)

Haz histogramas de la probabilidad de que una partícula del estado final sea un kaón o un pion.
Estos te ayudarán a orientarte sobre los valores de probabilidad adecuados para hacer los cortes.

También puedes considerar opciones más sofisticadas como gráficos 2D de probabilidades de kaón y pion, o diferentes valores de los cortes para las diferentes partículas del estado final.

In [ ]:
# plot the probability that a final state particle is a kaon

In [ ]:
# plot the probability that a final state particle is a pion

Ahora calcula la masa invariante del mesón B para los datos reales y traza un histograma de esto.
Compáralo con el que hiciste para los datos de simulación.

¿Puedes explicar las diferencias que observas?

In [ ]:
# draw a histogram for the B meson mass in the real data

Experimenta con los cortes y observa el impacto de cortes más severos o más indulgentes en el gráfico de masa invariante.
Debes seleccionar un conjunto de cortes que haga que la señal sea más prominente con respecto al fondo.
Una vez que hayas finalizado la selección sobre la identificación de partículas, también aplica cortes en la masa reconstruida de la partícula para seleccionar los eventos en el pico de masa del mesón B, eliminando los eventos de fondo que se encuentran en masas invariantes más bajas y más altas.

# Buscando diferencias globales entre materia y antimateria

En esta sección comenzarás a estudiar las diferencias entre materia y antimateria (violación CP). Aquí, "global" significa que estás buscando diferencias en todos los rangos de energía y momento (la cinemática) de los kaones en los que se han desintegrado los mesones B cargados. Más adelante veremos diferencias "locales" en diferentes regiones de la cinemática.

## Objetivos:
* Calcular la asimetría CP global
* Calcular la incertidumbre estadística
* Determinar si hay evidencia de violación de CP en esta desintegración

Para cuantificar la asimetría materia-antimateria en este proceso, deseamos comparar las partículas B<sup>+</sup> y B<sup>-</sup>. El B<sup>-</sup> es la antipartícula del B<sup>+</sup>.

¿Cómo puedes distinguir entre eventos que contienen partículas B<sup>+</sup> y B<sup>-</sup> usando `H1_Charge`, `H2_Charge` y `H3_Charge`?

In [ ]:
# make a variable for the charge of the B mesons

Ahora cuenta los números de eventos de cada uno de los dos tipos (N<sup>+</sup> y N<sup>-</sup>). También calcula la diferencia entre estos dos números.

In [ ]:
# make variables for the numbers of positive and negative B mesons

<!-- In order to calculate the Asymmetry, you can make use of the formula:
(note you may need to run this box in order to see the image)
<img src="https://drive.google.com/uc?id=1hl0Kt7Eg3hKP-bOL-O-qVJ9dncBDEAq-" width="200" /> -->
Para calcular la asimetría, puedes usar la fórmula:

<img src="opendata-project-1.2/Images/AsymmetryEq.png" width="200" />

In [ ]:
# calculate the value of the asymmetry, by using the formula above, and then print it

### Hint

**Diferenciando entre N+ y N-**

- La carga es una cantidad que se conserva. La carga del mesón $B$ es igual a la suma de las cargas de las partículas en las que se desintegra.

### Estimando la significancia de la desviación

Ahora necesitarás calcular la incertidumbre estadística de la asimetría. Puedes hacerlo usando la fórmula: 
<img src="opendata-project-1.2/Images/AsymmetryErrorEq.png" width="200" />

La significancia del resultado, sigma, se obtiene dividiendo el valor de la asimetría por su incertidumbre. Un valor que supere tres sigma se considera "evidencia" por los físicos de partículas, mientras que un valor de cinco sigma o más puede llamarse "observación" o "descubrimiento".

In [ ]:
# calculate the statistical significance of your result and print it

**¡Felicitaciones!** Has realizado tu primera búsqueda de una diferencia entre materia y antimateria.

Aquí solo has considerado la incertidumbre estadística. Tu medición también tendrá otras fuentes de incertidumbre conocidas como incertidumbres sistemáticas, que no has considerado en esta etapa.

# Diagramas de Dalitz y resonancias de dos cuerpos

## Objetivos:
* Producir diagramas de Dalitz de la muestra de datos simulados y reales
* Crear diagramas de Dalitz ordenados y binned (con intervalos)
* Identificar resonancias de dos cuerpos en los diagramas de Dalitz
____

En esta etapa te introducimos a una técnica importante para analizar desintegraciones de una partícula (tu mesón B cargado) en tres cuerpos (los tres kaones). Esto se conoce como diagrama de Dalitz.

La desintegración del mesón B puede ocurrir directamente al estado final de tres cuerpos o a través de una partícula intermedia. Por ejemplo, B<sup>+</sup> → K<sup>+</sup>K<sup>+</sup>K<sup>−</sup> podría ocurrir a través de la desintegración B<sup>+</sup> → K<sup>+</sup>R<sup>0</sup>, donde R<sup>0</sup> es una resonancia de partícula neutra que puede desintegrarse R<sup>0</sup> → K<sup>+</sup>K<sup>-</sup>. Los diagramas de Dalitz se pueden usar para identificar estas resonancias, que son visibles como bandas en el diagrama de Dalitz.

Puedes encontrar más información sobre estos diagramas y por qué se usan en la investigación en física de partículas en la [Introducción al Diagrama de Dalitz](opendata-project-1.2/Background-Information-Notebooks/DalitzPlots.ipynb).

La cinemática de una desintegración de tres cuerpos se puede describir completamente usando solo dos variables. Las energías y momentos de los tres kaones no son independientes entre sí porque todos provienen de la desintegración de un mesón B y la energía y el momento se conservan. Los ejes de los diagramas convencionalmente son las masas invariantes al cuadrado de dos pares de los productos de desintegración. Es un gráfico 2D, los ejes x e y son ambas masas al cuadrado y la densidad de puntos en el gráfico muestra la estructura.

Considera nuestra desintegración B<sup>+</sup> → K<sup>+</sup><sub>1</sub>K<sup>+</sup><sub>2</sub>K<sup>−</sup><sub>3</sub>, donde hemos numerado los kaones 1, 2, 3 para distinguirlos. Podemos calcular la masa invariante de tres combinaciones posibles que podrían corresponder a resonancias intermedias R<sup>++</sup><sub>1</sub> → K<sup>+</sup><sub>1</sub>K<sup>+</sup><sub>2</sub>, R<sup>0</sup><sub>2</sub> → K<sup>+</sup><sub>1</sub>K<sup>-</sup><sub>3</sub>, y R<sup>0</sup><sub>3</sub> → K<sup>+</sup><sub>3</sub>K<sup>-</sup><sub>3</sub>.

La potencial R<sup>++</sup><sub>1</sub> sería una resonancia doblemente cargada. No esperaríamos ver ninguna resonancia correspondiente a esto, ya que los mesones están compuestos por un quark y un antiquark y sus cargas no pueden sumar dos unidades.

Las potenciales R<sup>0</sup><sub>2</sub> y R<sup>0</sup><sub>3</sub> corresponden a configuraciones en las que podríamos ver resonancias. Por lo tanto, debes calcular las combinaciones de masa invariante para estas. El cuadrado de estas masas debe usarse como las variables de Dalitz.

Te sugerimos que hagas estos gráficos primero con los datos de simulación. En la simulación no hay resonancias intermedias y tu gráfico debería tener una densidad uniforme dentro del rango físicamente permitido por la conservación de energía y momento.

In [ ]:
# calculate the invariant masses for each possible hadron pair combination

In [ ]:
# plot the invariant mass for one of these combinations

In [ ]:
# make a Dalitz plot with labelled axes for the simulation data

### Hints

**Cálculo de la masa invariante** - Usa la misma técnica que usaste arriba para el mesón B, pero ahora aplicándola a masas invariantes de dos cuerpos en lugar de tres.

**Trazado del diagrama de Dalitz** - Puedes usar un gráfico de `scatter` (dispersión) de `matplotlib` para trazar un diagrama de Dalitz; consulta el [análisis de ejemplo](Example-Analysis.ipynb). Recuerda usar el cuadrado de cada masa de dos cuerpos.

## Añadiendo el diagrama de Dalitz para datos reales

Ahora dibuja un diagrama de Dalitz para los datos reales. Verifica que los signos de las cargas de los hadrones sean correctos para corresponder a tus potenciales resonancias neutras R<sup>0</sup><sub>2</sub> y R<sup>0</sup><sub>3</sub>.

In [ ]:
# calculate the invariant masses for each possible hadron pair combination in the real data

In [ ]:
# make a Dalitz plot for the real data (with your preselection cuts applied)

<div align="justify">Al dibujar el diagrama de Dalitz para los datos reales, etiqueta los ejes adecuadamente. Compara los diagramas de Dalitz de los datos reales con el de la simulación. ¿Cuáles son las diferencias más notables? </div>

### Ordenando las variables de Dalitz

Puedes hacer una mejora adicional para observar las resonancias más fácilmente. Tus resonancias R<sup>0</sup><sub>2</sub> y R<sup>0</sup><sub>3</sub> están compuestas por los mismos tipos de partículas, K<sup>+</sup>K<sup>-</sup>, y por lo tanto tienen las mismas distribuciones. Es útil imponer un orden que distinga las resonancias. Podemos llamar a las resonancias R<sup>0</sup><sub>Low</sub> y R<sup>0</sup><sub>High</sub>. En cada evento, R<sup>0</sup><sub>Low</sub> es la resonancia con la masa más baja y la otra corresponde a la combinación de kaones de mayor masa. Ahora puedes usar la masa de estas resonancias ordenadas como tus variables del diagrama de Dalitz, "plegando" efectivamente tu diagrama de Dalitz de modo que un eje siempre tenga un valor más alto que el otro.

In [ ]:
# make a new Dalitz plot with a mass ordering of the axes

### Hint

**Ordered Dalitz plot** - You can find the maximum of the mass of R<sup>0</sup><sub>Low</sub> vs R<sup>0</sup><sub>High</sub> elementwise on one axis, and the minimum of on the other. You can perform elementwise comparisons between two arrays  and return one array filled by either the individual min/max element from the elementwise comparisons.

### Binned Dalitz plot
You can improve the representation of your Dalitz plot by binning the data. The hist2d function can be used to make a 2D histogram. The number of bins specification in the hist2d function is the number of bins in one axis.

In [ ]:
# plot a binned Dalitz Plot
# use colorbar() to make a legend for your plot at the side

## Two body resonances

You can now use your Dalitz plot to identify the intermediate resonances that you see in your plots. The resonances will have shown up as bands of higher density of points on the plots. You can use the [particle data group](http://pdg.lbl.gov/2015/tables/contents_tables.html) tables of mesons to identify which particles these correspond to. The tables give the masses and widths of the particles and their decay modes. You are looking for mesons with the masses corresponding to where you see the bands and that decay into K<sup>+</sup>K<sup>-</sup>.

**Congratulations!** You have succesfully made a Dalitz plot and used it to observe the presence of intermediate particles in the decay of your charged B meson into three charged kaons.

# Searching for local matter anti-matter differences
## Aims:
* Observe matter antimatter differences (CP violation) in regions of the Dalitz plots of the B<sup>+</sup> and B<sup>-</sup> mesons.
* For the data in these regions produce plots to best display the CP violation.

In a section above you searched for global CP violation. You probably did not find a result with very high significance.

CP violation may arise from interference between decays through different resonances, and hence the magnitude and sign of the CP violation may vary across the Dalitz plot. We can apply the same equation as in the global CP violation study
<img src="https://drive.google.com/uc?id=1hl0Kt7Eg3hKP-bOL-O-qVJ9dncBDEAq-" width="200" />
but apply this only to events in particular regions of the Dalitz plot.


## Removing charm resonances

The analysis performed here is to study the CP violation in the charmless B meson decays to kaons. "charmless" means that the decay does not proceed through a charm quark. However, the most frequent decay of the B mesons occur through the *b* quark decaying into a *c* quark. The majority of these events can be removed by rejecting the events that are proceeding through a D<sup>0</sup> meson (which contains the charm quark).

In the section above you plotted a histogram of the invariant mass of the intermediate resonances and will have observed the D<sup>0</sup> meson in this and in the Dalitz plot. You should now reject events that are around the mass range of the D<sup>0</sup> meson to suppress this contribution. You can do this in your pre-selection on the data that you set-up earlier in the project.

This was also a simplification that we did not consider when we were calculating the global asymmetry. After you have applied this pre-selection your code will now recompute the global asymmetry with the D<sup>0</sup> meson contribution rejected. We have not yet observed CP violation in charm mesons and searching for this is another active area of current research.

## Comparing Dalitz plots

Make separate Dalitz plots for the B<sup>+</sup> and the B<sup>-</sup> decays.
Local CP Violation will show up as an asymmetry between the B<sup>+</sup> and the B<sup>-</sup> plots.  

In order that the statistical error on the asymmetry in each bin is not over large the bins need to contain a reasonable number of entries. Hence you will probably need larger bins than when you were looking for resonances in the section above. A suitable initial bin size might be $2.5~\text{GeV}^2/\text{c}^4 \times 2.5~\text{GeV}^2/\text{c}^4$.


In [ ]:
# make a Dalitz plot for the B+ events

In [ ]:
# make a Dalitz plot for the B- events

In [ ]:
# Make a plot showing the asymmetry between these two Daltz plots
# i.e. calculate the asymmetry between each bin of the B+ and B- Dalitz plots and show the result in another 2D plot

Observing a large asymmetry in some regions of the plot does not necessarily mean you have observed CP violation. If there are very few events in that region of the plot the uncertainty on that large asymmetry may be large. Hence, the value may still be compatible with zero.

You can calculate the statistical uncertainty on the asymmetry, for each bin of the plot, using the same formulas as you used in the global asymmetry section. You can then make a plot showing the uncertainty on the asymmetry.

Dividing the plot showing the asymmetry by the plot showing the statistical uncertainty you can then obtain the significance of the asymmetry in each bin. You can then plot the significance of the asymmetry to see if there is any evidence for CP violation.

In [ ]:
# Make a plot showing the uncertainty on the asymmetry

In [ ]:
# Make a plot showing the statistical significance of the asymmetry

## Observing CP violation

From your studies of the asymmetry plot, and the plot of its significance, you will be able to identify regions in the Dalitz plots that show indications of sizeable and significant CP Violation. You may find you have several consecutive bins with significant positive, or negative, asymmetries. You may wish to try alternative binnings of the Dalitz plots to best isolate the regions in which the significant asymmetries occur.

You can select events that are in these regions of the Dalitz plots where you observe signs of CP Violation. You can then plot a simple 1D histogram of the invariant mass distribution of the B<sup>+</sup> and the B<sup>-</sup> events, just as you did at the start of the project, but only for events that lie in the region of the Dalitz plot that you are interested in. Make the plots of the B<sup>+</sup> and the B<sup>-</sup> events with the same scale, or superimpose the two plots, so that you can observe if the particle and anti-particle decay processes are occurring at the same rate.

In [ ]:
# Make a plot showing the invariant mass of the B+ meson particles
# using events from a region of the Dalitz plot showing sizeable CP asymmetries

In [ ]:
# Make a plot showing the invariant mass of the B- meson particles using events from the same region

**Congratulations!** You should now have succesfully observed significant evidence for CP Violation. You should have plots that clearly show that particle and anti-particle decay processes occur at different rates in local regions of the Dalitz plot. You may wish to comapre your results with those published by the LHCb collaboration in this [paper](http://lhcbproject.web.cern.ch/lhcbproject/Publications/LHCbProjectPublic/LHCb-PAPER-2013-027.html).

**Well Done** you have succesfully completed your first particle physics analysis project. There are many more analyses that can be conducted witht the data set that you have been provided and the skills that you have gained. Ideas for some of these are explored in the section below. Maybe you can discover something new!

## Adding extra sophistication

### Fitting the mass
So far you used only the number of total events to calculate the asymmetry. However, your invariant mass histogram is a combination of signal sitting on top of background. To extract the correct number of signal candidates you have to use a function to fit your signal and background.

### Hint - Challenges 3 and 4
Some typical functions for the signal are the Gaussian or the Crystal ball funtions. For the background, the exponential function or a polynomial are commonly used.

In [ ]:
# Plot and fit the invariant mass of the B-meson

In [ ]:
# Calculate the CP asymmetry using the values extracted from the fit

### Systematic Uncertainties - Challenge 4
In this analysis you considered the statistical uncertainty on the result. This occurs as a result of having only a limited number of events. In addition there are [systematic uncertainties](https://en.wikipedia.org/wiki/Observational_error#Systematic_versus_random_error), these arise from biases in your measurement.

In this section you will compute the systematic uncertainties of using diferent models/functions to fit your signal and background. Try using a different function for your signal (e.g. instead of a Gaussian use a Crystal Ball) and compute the asymmetry when using this other function. Do the same for the background function.

The systematic uncertainty will be the difference between the two values of the asymmetry.

In [ ]:
# Plot and fit the invariant mass of the B-meson using another fit function

In [ ]:
# Calculate the CP asymmetry using the values extracted from the fit

In [ ]:
# Compute the systematic uncertainty

# Further analyses

The data set you have been provided is the full set of data recorded by LHCb preselected for decays of charged B mesons into three final state tracks. This data set has been used for two important publications, [here](http://lhcbproject.web.cern.ch/lhcbproject/Publications/LHCbProjectPublic/LHCb-PAPER-2013-027.html) and [here](http://lhcbproject.web.cern.ch/lhcbproject/Publications/LHCbProjectPublic/LHCb-PAPER-2013-051.html).

We discuss here:
<ul>
<li>Additional elements that you could add to your analysis of B<sup>+</sup> → K<sup>+</sup>K<sup>+</sup>K<sup>−</sup> </li>
<li>Further analyses that you could perform with this data set</li>
</ul>

### More Systematic Uncertainties
In this analysis you considered the statistical uncertainty on the result. This occurs as a result of having only a limited number of events. In addition there are [systematic uncertainties](https://en.wikipedia.org/wiki/Observational_error#Systematic_versus_random_error), these arise from biases in your measurement. Here we discuss three sources of these for this analysis.
<ul>
<li> Production asymmetry. The LHC is a proton-proton collider and hence the initial state of the collision is not matter antimatter symmetric. Consequently B<sup>+</sup> and B<sup>-</sup> mesons may not be produced at exactly the same rates. This small production asymmetry it is estimated could be approximately 1%. It can also be measured from the data, as discussed in the LHCb paper.</li>
<li> Detection asymmetry. The LHCb detector could be more efficient for detecting either the B<sup>+</sup> or the B<sup>-</sup> final states. This is because the positive and negative kaons will be bent by the magnet indifferent directions in the detector. If the efficiency of the detector is higher in one region than another this will lead to higher efficiencies for K<sup>+</sup> or K<sup>-</sup> and hence for B<sup>+</sup> or B<sup>-</sup>. For this reason the magnetic field of the LHCb detector is regularly reversed. You used data in this analysis in which the magnetic field was both up and down and hence the effect will (partially) cancel. By comparing results for the two magnet polarities separately you can check the size of this effect. When loading the data above both polarities were combined, you can instead load them independently to measure the difference between the two datasets.</li>
<li> Analysis technique. The analysis technique you have used may bias the result. A major simplification we made in the analysis above was to neglect 'background' events. We imposed a selection to select a sample of predominantly signal events but have not accounted for the effect of the residual background events.</li>
</ul>

### Using mass sidebands

One source of 'background' events arises from random combinations of tracks in events that happen to fake the 'signal' characteristics. These events will not peak in the mass distribution at the mass of the B meson but rtaher will have a smoothly varying distribution. Looking at the number and distribution of of events away from the mass peak can allow you to estimate the number of background events under the mass peak.

## Further analyses

The LHCb papers using this data set that you are using analysed four decay channels of the charged B mesons. You can perform any of these analyses.
<ul>
<li>B<sup>+</sup> → K<sup>+</sup>K<sup>+</sup>K<sup>−</sup> (and anti-particle equivalent). This is the analysis you have performed here. It has the lowest background of the four channels and hence the approximation we made of neglecting the background events will give the least bias to this channel.</li>
<li>B<sup>+</sup> → &pi;<sup>+</sup>&pi;<sup>+</sup>&pi;<sup>−</sup> (and anti-particle equivalent). In this analysis the final state is three charged pions. The level of background events compared to the signal is significantly higher as pions are the most commonly produced particle at the LHC. Hence, a method of estimating the background level should be added to complete this analysis.</li>
<li>B<sup>+</sup> → K<sup>+</sup>&pi;<sup>+</sup>&pi;<sup>−</sup> (and anti-particle equivalent). In this analysis the final state is a mixture of one kaon and two pions. This means that the analysis needs to determine in each event which track is the best candidate kaon and apply selection cuts appropriately to select out the events.</li>
<li>B<sup>+</sup> → &pi;<sup>+</sup>K<sup>+</sup>K<sup>−</sup> (and anti-particle equivalent). This channel has a higher level of background compared to the signal.</li>
</ul>